## Loading Packages

In [2]:
import pandas as pd
from pathlib import Path

## Data Path

In [3]:
DATA_DIR = Path("../data")
csv_path = next(DATA_DIR.glob("*.csv"))
print("File:", csv_path.name)

df = pd.read_csv(csv_path)
df.head()

File: Unicorn_Companies.csv


,Company,Valuation,Date Joined,Industry,City,Country,Continent,Year Founded,Funding,Select Investors
0,Bytedance,$180B,2017-04-07,Artificial intelligence,Beijing,China,Asia,2012,$8B,"Sequoia Capital China, SIG Asia Investments, S..."
1,SpaceX,$100B,2012-12-01,Other,Hawthorne,United States,North America,2002,$7B,"Founders Fund, Draper Fisher Jurvetson, Rothen..."
2,SHEIN,$100B,2018-07-03,E-commerce & direct-to-consumer,Shenzhen,China,Asia,2008,$2B,"Tiger Global Management, Sequoia Capital China..."
3,Stripe,$95B,2014-01-23,Fintech,San Francisco,United States,North America,2010,$2B,"Khosla Ventures, LowercaseCapital, capitalG"
4,Klarna,$46B,2011-12-12,Fintech,Stockholm,Sweden,Europe,2005,$4B,"Institutional Venture Partners, Sequoia Capita..."


In [4]:
df.shape

(1074, 10)

## Data Overview

In [5]:
df_columns = (df.columns.str.strip().str.lower().str.replace(r"[^a-z0-9]+", "_", regex=True)
              .str.strip("_"))
print(df.columns.tolist())
df.info()
print(df.isna().sum())
print("Dublicates", df.duplicated().sum())
df.head()

['Company', 'Valuation', 'Date Joined', 'Industry', 'City', 'Country', 'Continent', 'Year Founded', 'Funding', 'Select Investors']
<class 'pandas.DataFrame'>
RangeIndex: 1074 entries, 0 to 1073
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Company           1074 non-null   str  
 1   Valuation         1074 non-null   str  
 2   Date Joined       1074 non-null   str  
 3   Industry          1074 non-null   str  
 4   City              1058 non-null   str  
 5   Country           1074 non-null   str  
 6   Continent         1074 non-null   str  
 7   Year Founded      1074 non-null   int64
 8   Funding           1074 non-null   str  
 9   Select Investors  1073 non-null   str  
dtypes: int64(1), str(9)
memory usage: 84.0 KB
Company              0
Valuation            0
Date Joined          0
Industry             0
City                16
Country              0
Continent            0
Year Founded         0
Fu

,Company,Valuation,Date Joined,Industry,City,Country,Continent,Year Founded,Funding,Select Investors
0,Bytedance,$180B,2017-04-07,Artificial intelligence,Beijing,China,Asia,2012,$8B,"Sequoia Capital China, SIG Asia Investments, S..."
1,SpaceX,$100B,2012-12-01,Other,Hawthorne,United States,North America,2002,$7B,"Founders Fund, Draper Fisher Jurvetson, Rothen..."
2,SHEIN,$100B,2018-07-03,E-commerce & direct-to-consumer,Shenzhen,China,Asia,2008,$2B,"Tiger Global Management, Sequoia Capital China..."
3,Stripe,$95B,2014-01-23,Fintech,San Francisco,United States,North America,2010,$2B,"Khosla Ventures, LowercaseCapital, capitalG"
4,Klarna,$46B,2011-12-12,Fintech,Stockholm,Sweden,Europe,2005,$4B,"Institutional Venture Partners, Sequoia Capita..."


## Cleaning

In [14]:
VAL_COL = "valuation_b"  
FUND_COL = "total_raised"    

In [15]:
df = pd.read_csv(csv_path)   

df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))

df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))

def to_billions(x):
    """'$1.5B' -> 1.5, '$800M' -> 0.8, 'Unknown' -> NaN"""
    if pd.isna(x):
        return float("nan")
    s = str(x).replace("$", "").replace(",", "").strip()
    if not s or s.lower() == "unknown":
        return float("nan")
    mult = {"B": 1, "M": 1/1000, "K": 1/1_000_000}
    if s[-1].upper() in mult:
        return float(s[:-1]) * mult[s[-1].upper()]
    return float(s)

df = df.drop_duplicates().copy()
df["valuation_b"] = df["valuation"].apply(to_billions)
df["funding_b"] = df["funding"].apply(to_billions)

# 3. Ամսաթվեր
df["date_joined"] = pd.to_datetime(df["date_joined"], errors="coerce")
df["year_joined"] = df["date_joined"].dt.year

# 4. Նոր սյունակներ
df["years_to_unicorn"] = df["year_joined"] - df["year_founded"]
df["valuation_per_funding"] = df["valuation_b"] / df["funding_b"].where(df["funding_b"] > 0)

# 5. Ներդրողների ցուցակ
df["investors_list"] = df["select_investors"].fillna("").str.split(",").apply(
    lambda lst: [i.strip() for i in lst if i.strip()]
)

print(df.columns.tolist())
df[["company", "valuation_b", "funding_b", "year_joined", "years_to_unicorn"]].describe()

['company', 'valuation', 'date_joined', 'industry', 'city', 'country', 'continent', 'year_founded', 'funding', 'select_investors', 'valuation_b', 'funding_b', 'year_joined', 'years_to_unicorn', 'valuation_per_funding', 'investors_list']


,valuation_b,funding_b,year_joined,years_to_unicorn
count,1074.000000,1062.000000,1074.000000,1074.000000
mean,3.455307,0.557269,2019.896648,7.000931
std,8.547022,0.810134,2.001288,5.329672
min,1.000000,0.000000,2007.000000,-4.000000
25%,1.000000,0.223000,2019.000000,4.000000
50%,2.000000,0.370000,2021.000000,6.000000
75%,3.000000,0.609250,2021.000000,9.000000
max,180.000000,14.000000,2022.000000,98.000000


In [11]:
print("Valuation NaN:", df["valuation_b"].isna().sum())
print("Funding NaN:", df["funding_b"].isna().sum())
print("Date NaN:", df["date_joined"].isna().sum())
print("Բացասական years_to_unicorn:", (df["years_to_unicorn"] < 0).sum())
df.loc[df["years_to_unicorn"] < 0, ["company", "year_founded", "year_joined"]]

Valuation NaN: 0
Funding NaN: 12
Date NaN: 0
Բացասական years_to_unicorn: 1


,company,year_founded,year_joined
714,Yidian Zixun,2021,2017


## Saving

In [16]:
clean_path = Path("../data/unicorn_clean.csv")
df.drop(columns="investors_list").to_csv(clean_path, index=False)
print("Saved:", clean_path, df.shape)

Saved: ..\data\unicorn_clean.csv (1074, 16)
